# Validation comparison and frozen configuration

Lead 0 is separate; leads 1–6 are primary, 12/24 are secondary. Tables use latest run_id per (model,fold,lead); stale records remain in the append-only log. Models fit TRAIN only; feature gates and candidate/ensemble selection use walk-forward VALID. Seven-day paired UTC blocks cover all regions together. CIs do not adjust for hyperparameter selection/multiple comparisons. No TEST or remaining optional experiment is run.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
root = Path.cwd() if Path("src/airalerts").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from airalerts.evaluate import latest_results
out = root / "reports/model"
latest = latest_results(out / "results.csv")


In [2]:
comparison = pd.read_csv(out / "comparison_table.csv")
for reference in ["clim", "persistence", "ses"]:
    table = comparison.loc[comparison.reference.eq(reference)]
    print("BSS versus", reference)
    print(table.pivot(index="model", columns="lead", values="bss").to_string())
    print(table[["model", "lead", "bss", "ci_low", "ci_high", "ece"]].to_string(index=False))


BSS versus clim
lead                          0         1         2         3         4         5         6         12        24
model                                                                                                           
climatology             0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000
ens_ses_hgb             0.427908       NaN       NaN       NaN       NaN       NaN       NaN       NaN  0.044785
ens_ses_logreg_ABL           NaN  0.110486  0.075344  0.067700       NaN       NaN  0.054112  0.047298       NaN
ens_ses_logreg_AB_hl30       NaN       NaN       NaN       NaN  0.063291  0.058340       NaN       NaN       NaN
hgb                     0.438384  0.102931  0.064397  0.057759  0.050946  0.042314  0.030049  0.033725  0.032412
logreg_A                0.436156  0.090061  0.054536  0.047078  0.042923  0.037034  0.030693  0.017121  0.009744
logreg_AB               0.438425  0.099399  0.065196  0.057834  0.048987  0.0437

![BSS versus forecast lead](../reports/model/bss_vs_lead.png)

All recorded models are shown; lead 0 is a separate panel and is never averaged with primary leads.


In [3]:
frozen = json.loads((root / "config/final_config.json").read_text())
print(pd.read_csv(out / "frozen_validation.csv").to_string(index=False))
print(pd.read_csv(out / "ensemble_selection.csv").to_string(index=False))
print("Frozen status:", frozen["status"], "TEST evaluated:", frozen["test_evaluated"])


 lead                  model     n    brier  bss_vs_clim  bss_vs_persistence  bss_vs_ses      ece  half_life  ensemble_weight
    0                    hgb 35136 0.054818     0.438384            0.042622    0.348814 0.011552        inf              NaN
    1     ens_ses_logreg_ABL 35112 0.086834     0.110486            0.053692    0.056329 0.015771       14.0             0.75
    2     ens_ses_logreg_ABL 35088 0.090281     0.075344            0.055538    0.033498 0.015970       14.0             0.75
    3     ens_ses_logreg_ABL 35064 0.091034     0.067700            0.053382    0.024202 0.017451       30.0             0.75
    4 ens_ses_logreg_AB_hl30 35040 0.091466     0.063291            0.048308    0.011789 0.012716       30.0             0.50
    5 ens_ses_logreg_AB_hl30 35016 0.091926     0.058340            0.044952    0.008687 0.012423       30.0             0.50
    6     ens_ses_logreg_ABL 34992 0.092342     0.054112            0.043792    0.005897 0.010112       14.0          

## Per-oblast metrics and TRAIN flags

Flags are fold/lead-specific TRAIN target rates below 3% or above 95%; base_rate in the result schema is the evaluation positive rate. This table keeps folds explicit rather than assigning one retrospective flag to all periods.


In [4]:
regions = pd.read_csv(out / "best_per_oblast.csv")
print(regions.to_string(index=False))


                 model  lead  fold_id                  oblast  flagged   n  base_rate    brier  bss_vs_clim  bss_vs_persistence    bss_vs_ses      ece
                   hgb     0        1        Cherkaska oblast    False 360   0.141667 0.067998     0.467886           -0.005466  4.151137e-01 0.036293
                   hgb     0        1     Chernihivska oblast    False 360   0.925000 0.043483     0.545503            0.191873  3.163747e-01 0.010800
                   hgb     0        1     Chernivetska oblast    False 360   0.025000 0.011346     0.537947           -0.029809  4.532954e-01 0.013515
                   hgb     0        1  Dnipropetrovska oblast     True 360   0.994444 0.006659     0.068163            0.104794  7.900559e-02 0.011674
                   hgb     0        1         Donetska oblast    False 360   0.983333 0.010778     0.394207            0.057918  3.812648e-01 0.011403
                   hgb     0        1 Ivano-Frankivska oblast    False 360   0.002778 0.003604

![Frozen-model validation calibration](../reports/model/best_calibration.png)

The frozen settings contain per-lead model, ordered features, hyperparameters, half-life and learned-model ensemble weight. Future FINAL fitting must estimate coefficients/SES alpha/HGB stopping iterations using only FINAL TRAIN and the frozen rules.

See [validation_summary.md](../reports/model/validation_summary.md) for the ordered experiments and HYPOTHESES NOT CONFIRMED. TEST remains unevaluated.
